# 03 - Resultados de scoring | RADAR Cibest

**Fase ASUM-DM:** 4 - Modelado  
**Objetivo:** Ejecutar el scoring hibrido completo y revisar resultados

In [818]:
import sys
from pathlib import Path
import importlib
import re

# Asegura que Python encuentre el proyecto
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

# Importar paquete raíz y módulos que quieres recargar
import src
import src.utils as utils
import src.data_preparation.feature_engineering as feature_engineering
import src.scoring.hybrid_scorer as hybrid_scorer
import src.scoring.ranking as ranking
import src.scoring.explainability as explain_scorer


# Invalidar cachés de importación
importlib.invalidate_caches()

# Recargar módulos modificados
importlib.reload(utils)
importlib.reload(feature_engineering)
importlib.reload(ranking)
importlib.reload(hybrid_scorer)
importlib.reload(explain_scorer)

# Reimportar funciones después del reload
from src.utils import load_all_configs, resolve_data_path, setup_logger, get_variable_catalog
from src.scoring.hybrid_scorer import run_full_scoring, prepare_decision_matrix
from src.scoring.explainability import compute_all_business_line_contributions, build_explainability_table_for_line, get_top_contributors, get_top_shortfalls, summarize_contributions_by_dimension, generate_country_line_explanation, compare_country_across_lines, compare_countries_in_line, compute_all_marginal_effects, combine_contribution_and_marginal, classify_driver_robustness, build_country_driver_table

# Cargar configuración
configs = load_all_configs()
setup_logger(configs["settings"].get("logging"))

# ---------------------------------------------------------------------
# Cargar master latest_available correcto
# ---------------------------------------------------------------------
raw_dir = resolve_data_path(configs["settings"]["data"]["raw_path"])

pattern = re.compile(r"^master_raw_\d{8}\.parquet$")

master_files = sorted(
    [
        p for p in raw_dir.glob("master_raw_*.parquet")
        if pattern.match(p.name)
    ],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not master_files:
    raise FileNotFoundError(
        "Falta master_raw_YYYYMMDD.parquet. Ejecute primero notebook 01."
    )

master_path = master_files[0]
master = pd.read_parquet(master_path)

print(f"Archivo master cargado: {master_path.name}")
print(f"Master cargado: {master.shape}")
print(f"Variables únicas: {master['variable'].nunique()}")
print("Tiene gdp_growth:", "gdp_growth" in master["variable"].unique())

# ---------------------------------------------------------------------
# Validaciones defensivas
# ---------------------------------------------------------------------
required_cols = {"country_iso3", "year", "variable", "value", "source"}
missing_cols = required_cols - set(master.columns)

if missing_cols:
    raise ValueError(
        f"Master inválido. Faltan columnas: {missing_cols}. "
        f"Columnas actuales: {master.columns.tolist()}"
    )

master["variable"] = master["variable"].astype(str).str.strip()

if "gdp_growth" not in master["variable"].unique():
    raise ValueError(
        "El master cargado no contiene gdp_growth. "
        "Probablemente se cargó un master histórico/parcial equivocado."
    )

if master["variable"].nunique() < 35:
    raise ValueError(
        f"El master cargado tiene solo {master['variable'].nunique()} variables. "
        "Para el scoring actual se esperaba un master completo con cerca de 44 variables."
    )

# Limpieza defensiva por si algún master anterior quedó con wgi_composite
master = master[master["variable"] != "wgi_composite"].copy()

# ---------------------------------------------------------------------
# Validar matriz de decisión
# ---------------------------------------------------------------------
wide_raw, decision_matrix, excluded = prepare_decision_matrix(master, configs)

print(f"wide_raw shape: {wide_raw.shape}")
print(f"decision_matrix shape: {decision_matrix.shape}")
print("wgi_composite en decision_matrix:", "wgi_composite" in decision_matrix.columns)
print("gdp_growth en master:", "gdp_growth" in master["variable"].unique())

# ---------------------------------------------------------------------
# Ejecutar scoring
# ---------------------------------------------------------------------
results = run_full_scoring(master, configs, persist=True)

print(f"Paises evaluados: {len(results['radar_global'])}")
print(f"Excluidos por cobertura: {results['excluded_countries']}")

print(f"País origen: {results['origin_country']}")
print(f"País origen excluido: {results['origin_country_excluded']}")

# print("\nResumen tendencia:")
# print(results["trend"]["trend"].describe())
# print(results["trend"].sort_values("trend", ascending=False).head(10))

2026-06-12 21:40:52 | INFO     | src.data_preparation.cleaning:pivot_long_to_wide:70 | Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)
2026-06-12 21:40:52 | WARNING  | src.data_preparation.cleaning:validate_country_coverage:92 | Paises excluidos por >30% variables faltantes: ['CUB']


Archivo master cargado: master_raw_20260612.parquet
Master cargado: (1281, 5)
Variables únicas: 45
Tiene gdp_growth: True


2026-06-12 21:40:53 | INFO     | src.data_preparation.cleaning:impute_missing:145 | Imputacion (regional_median): 107 -> 0 celdas faltantes
2026-06-12 21:40:53 | INFO     | src.data_preparation.cleaning:run_cleaning:183 | Limpieza completada: 29 paises x 45 variables
2026-06-12 21:40:53 | INFO     | src.data_preparation.feature_engineering:apply_log_transform:73 | Log-transformacion (natural) aplicada a: ['gdp_nominal', 'population_total', 'geographic_distance_km', 'colombian_diaspora_stock', 'stock_market_cap_gdp', 'financial_system_deposits_gdp', 'domestic_credit_private_gdp', 'fdi_net_inflows_gdp', 'personal_remittances_gdp', 'secure_internet_servers_per_million', 'atms_per_100k_adults', 'commercial_bank_branches_per_100k_adults']
2026-06-12 21:40:53 | INFO     | src.data_preparation.feature_engineering:run_feature_engineering:136 | Feature engineering completado: 29 paises x 46 variables
2026-06-12 21:40:53 | INFO     | src.scoring.hybrid_scorer:prepare_decision_matrix:109 | Variab

wide_raw shape: (29, 46)
decision_matrix shape: (29, 35)
wgi_composite en decision_matrix: False
gdp_growth en master: True


2026-06-12 21:40:55 | INFO     | src.data_preparation.cleaning:impute_missing:145 | Imputacion (regional_median): 107 -> 0 celdas faltantes
2026-06-12 21:40:55 | INFO     | src.data_preparation.cleaning:run_cleaning:183 | Limpieza completada: 29 paises x 45 variables
2026-06-12 21:40:55 | INFO     | src.data_preparation.feature_engineering:apply_log_transform:73 | Log-transformacion (natural) aplicada a: ['gdp_nominal', 'population_total', 'geographic_distance_km', 'colombian_diaspora_stock', 'stock_market_cap_gdp', 'financial_system_deposits_gdp', 'domestic_credit_private_gdp', 'fdi_net_inflows_gdp', 'personal_remittances_gdp', 'secure_internet_servers_per_million', 'atms_per_100k_adults', 'commercial_bank_branches_per_100k_adults']
2026-06-12 21:40:55 | INFO     | src.data_preparation.feature_engineering:run_feature_engineering:136 | Feature engineering completado: 29 paises x 46 variables
2026-06-12 21:40:55 | INFO     | src.scoring.hybrid_scorer:prepare_decision_matrix:109 | Variab

Paises evaluados: 28
Excluidos por cobertura: ['CUB']
País origen: COL
País origen excluido: True


In [819]:

print("gdp_growth en master:", "gdp_growth" in master["variable"].unique())
print("gdp_growth en wide_raw:", "gdp_growth" in wide_raw.columns)
print("gdp_growth en decision_matrix:", "gdp_growth" in decision_matrix.columns)

gdp_growth en master: True
gdp_growth en wide_raw: True
gdp_growth en decision_matrix: False


In [820]:
"gdp_growth" in configs["weights"]["variable_weights"]["macro"]

False

## Top 10 mercados (RADAR global)

In [821]:
#radar_global representa el score RADAR compuesto para cada país, ordenado de mayor a menor. 
# Este score es el resultado final del modelo híbrido que combina los diferentes factores evaluados en el análisis. 
# Al mostrar las primeras filas de este DataFrame, puedes ver cuáles son los países mejor posicionados según el modelo 
# y sus respectivos scores.
results['radar_global'].round(3) 

,country_iso3,radar_score,rank
0,PAN,0.723,1
1,CRI,0.660,2
2,DOM,0.634,3
3,ESP,0.627,4
4,CHL,0.592,5
5,ECU,0.583,6
6,PER,0.559,7
7,MEX,0.559,8
8,GTM,0.548,9
9,VEN,0.547,10


In [822]:
# global_ranking representa el ranking TOPSIS puro.
results['global_ranking']#.head(10).round(3)

,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank
country_iso3,,,,,,,,
CAN,0.719156,0.060493,0.154904,0.663832,0.649818,0.928552,0.645724,1
USA,0.717864,0.061544,0.156592,0.692189,0.661877,0.829612,0.725722,2
ESP,0.665248,0.068178,0.135489,0.608580,0.611979,0.782554,0.723244,3
CHL,0.636569,0.075143,0.131617,0.541781,0.577081,0.813094,0.667066,4
URY,0.589495,0.086857,0.124730,0.458270,0.486805,0.855351,0.631845,5
CRI,0.581929,0.084705,0.117904,0.478381,0.505199,0.775447,0.579880,6
BHS,0.557503,0.090649,0.114209,0.391290,0.564119,0.737779,0.526366,7
PAN,0.557322,0.086909,0.109417,0.482605,0.549518,0.623587,0.583590,8
DOM,0.549547,0.090138,0.109967,0.482665,0.534937,0.643661,0.468258,9


## Ranking por linea de negocio

In [823]:
results.keys()

dict_keys(['decision_matrix', 'wide_raw', 'global_ranking', 'business_line_rankings', 'ipc', 'trend', 'radar_global', 'radar_by_line', 'excluded_countries', 'origin_country', 'origin_country_excluded', 'composite_weights'])

In [824]:
results["business_line_rankings"].keys()

dict_keys(['IB', 'PF', 'AD', 'BD', 'CIB'])

In [825]:
results["business_line_rankings"]["IB"]#.head(10).round(3) #ver una línea específica

,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
country_iso3,,,,,,,,,
CAN,0.784153,0.054338,0.197404,0.662985,0.774630,0.936147,0.679722,1,IB
USA,0.735974,0.064208,0.178980,0.610650,0.739453,0.836186,0.759616,2,IB
ESP,0.709253,0.069856,0.170408,0.572680,0.715769,0.776457,0.754096,3,IB
CHL,0.677494,0.075654,0.158929,0.621747,0.652477,0.789307,0.698967,4,IB
BHS,0.625508,0.085281,0.142443,0.482449,0.635704,0.692603,0.580584,5,IB
URY,0.624730,0.087105,0.145008,0.523585,0.573514,0.825400,0.670617,6,IB
CRI,0.608649,0.090426,0.140636,0.579099,0.567710,0.740949,0.601370,7,IB
JAM,0.604456,0.091757,0.140220,0.439036,0.656483,0.594806,0.497629,8,IB
BRB,0.596921,0.102965,0.152481,0.397711,0.593200,0.774293,0.436684,9,IB


In [826]:
df_view = results["radar_by_line"].copy() #ver una línea específica.copy()

score_candidates = ["score", "topsis_score", "closeness", "radar_score"]
rank_candidates = ["rank", "ranking", "position"]

score_col = next((c for c in score_candidates if c in df_view.columns), None)
rank_col = next((c for c in rank_candidates if c in df_view.columns), None)

format_dict = {}

if score_col:
    format_dict[score_col] = "{:.3f}"

if rank_col:
    format_dict[rank_col] = "{:.0f}"

styler = df_view.style.format(format_dict)

if score_col:
    styler = styler.background_gradient(
        subset=[score_col],
        cmap="RdYlGn"
    )

if rank_col:
    styler = styler.background_gradient(
        subset=[rank_col],
        cmap="RdYlGn_r"
    )

styler.format(precision=3)

,country_iso3,IB,PF,AD,BD,CIB,GLOBAL,rank_global
19,PAN,0.746,0.698,0.717,0.723,0.703,0.723,1
8,CRI,0.676,0.639,0.675,0.681,0.593,0.660,2
9,DOM,0.619,0.604,0.606,0.630,0.629,0.634,3
11,ESP,0.653,0.629,0.655,0.678,0.622,0.627,4
7,CHL,0.616,0.578,0.625,0.641,0.602,0.592,5
10,ECU,0.607,0.569,0.536,0.594,0.539,0.583,6
20,PER,0.548,0.541,0.537,0.576,0.549,0.559,7
17,MEX,0.527,0.510,0.515,0.563,0.540,0.559,8
12,GTM,0.556,0.494,0.456,0.475,0.518,0.548,9
27,VEN,0.585,0.616,0.523,0.626,0.511,0.547,10


## Indice de Proximidad Compuesto

In [827]:
results['ipc'][['ipc']].round(3)

,ipc
country_iso3,
ECU,0.997
PAN,0.981
VEN,0.892
DOM,0.854
CRI,0.847
PER,0.810
BOL,0.801
SLV,0.753
NIC,0.742


## Tendencia

In [828]:
#validar fuente de calculo:
configs["settings"]["scoring"]["trend"]

{'enabled': True,
 'method': 'gdp_growth',
 'variable': 'gdp_growth',
 'end_year': 2024,
 'n_years': 3,
 'neutral_value': 0.5,
 'winsor_lower_quantile': 0.05,
 'winsor_upper_quantile': 0.95,
 'risk_adjusted': False,
 'risk_variable': 'country_risk_premium',
 'risk_floor': 0.5}

In [829]:
results["trend"].sort_values("trend", ascending=False).round(3)

,country_iso3,trend
5,BRB,1.000
13,GUY,1.000
19,PAN,0.941
1,BHS,0.744
27,VEN,0.744
8,CRI,0.565
2,BLZ,0.526
9,DOM,0.478
11,ESP,0.473
18,NIC,0.434


In [830]:
results["trend"].describe()

,trend
count,28.000000
mean,0.379298
std,0.289187
min,0.000000
25%,0.153660
50%,0.309718
75%,0.490212
max,1.000000


In [831]:
#Esto mostrará qué países están recibiendo mayor impulso por momentum macroeconómico.
audit = (
    results["radar_global"]
    .merge(results["trend"], on="country_iso3", how="left")
    .sort_values("trend", ascending=False)
)

audit#.head(10)

,country_iso3,radar_score,rank,trend
20,BRB,0.489188,21,1.000000
21,GUY,0.479533,22,1.000000
0,PAN,0.722897,1,0.941349
19,BHS,0.491199,20,0.744282
9,VEN,0.546586,10,0.744162
1,CRI,0.659813,2,0.564922
25,BLZ,0.425406,26,0.525522
2,DOM,0.633875,3,0.478441
3,ESP,0.626675,4,0.473182
15,NIC,0.506170,16,0.434257


In [832]:
results["composite_weights"]

{'alpha': 0.6, 'beta': 0.3, 'gamma': 0.1}

In [833]:
# 1. TOPSIS (score base)
global_rank = results["global_ranking"].copy()

if "country_iso3" in global_rank.columns:
    global_rank = global_rank.set_index("country_iso3")

topsis = global_rank["score"].rename("topsis_score")


# 2. IPC
ipc_df = results["ipc"].copy()

if "country_iso3" in ipc_df.columns:
    ipc = ipc_df.set_index("country_iso3")["ipc"]
else:
    ipc = ipc_df["ipc"]


# 3. Trend
trend_df = results["trend"].copy()

if "country_iso3" in trend_df.columns:
    trend = trend_df.set_index("country_iso3")["trend"]
else:
    trend = trend_df["trend"]


# 4. Unir todo
df = pd.concat([topsis, ipc, trend], axis=1)


# 5. Manejo de faltantes (igual que el modelo)
df["ipc"] = df["ipc"].fillna(df["ipc"].median())
df["trend"] = df["trend"].fillna(df["trend"].median())


# 6. Pesos
alpha = results["composite_weights"]["alpha"]
beta = results["composite_weights"]["beta"]
gamma = results["composite_weights"]["gamma"]



# 7. Aportes
df["aporte_topsis"] = alpha * df["topsis_score"]
df["aporte_ipc"] = beta * df["ipc"]
df["aporte_trend"] = gamma * df["trend"]


# 8. Score final
df["radar_score"] = (
    df["aporte_topsis"] +
    df["aporte_ipc"] +
    df["aporte_trend"]
)


# 9. Rankings comparativos
df["rank_topsis"] = df["topsis_score"].rank(ascending=False, method="min").astype(int)
df["rank_radar"] = df["radar_score"].rank(ascending=False, method="min").astype(int)

df["delta_rank"] = df["rank_topsis"] - df["rank_radar"]


# 10. Orden final
df = df.sort_values("rank_radar")

df#.head(20)

,topsis_score,ipc,trend,aporte_topsis,aporte_ipc,aporte_trend,radar_score,rank_topsis,rank_radar,delta_rank
country_iso3,,,,,,,,,,
PAN,0.557322,0.981231,0.941349,0.334393,0.294369,0.094135,0.722897,8,1,7
CRI,0.581929,0.847213,0.564922,0.349157,0.254164,0.056492,0.659813,6,2,4
DOM,0.549547,0.854342,0.478441,0.329728,0.256303,0.047844,0.633875,9,3,6
ESP,0.665248,0.600693,0.473182,0.399149,0.180208,0.047318,0.626675,3,4,-1
CHL,0.636569,0.666281,0.096829,0.381941,0.199884,0.009683,0.591509,4,5,-1
ECU,0.451550,0.997082,0.125770,0.270930,0.299125,0.012577,0.582632,22,6,16
PER,0.508048,0.809513,0.117901,0.304829,0.242854,0.011790,0.559473,17,7,10
MEX,0.529872,0.713784,0.268134,0.317923,0.214135,0.026813,0.558872,12,8,4
GTM,0.480442,0.725909,0.423651,0.288265,0.217773,0.042365,0.548403,19,9,10


## 1. ¿Por qué sube un país?
Ejemplo típico (Panamá):

Tiene buen ipc
Tiene buen trend
Aunque su topsis_score no sea top

--> Resultado: sube en rank_radar

## 2. ¿Quién “pierde” cuando se mete IPC y Trend?
Países como:

USA
CAN

Muchas veces bajan porque:

Aunque son fuertes en TOPSIS
No necesariamente tienen la mejor proximidad (IPC)

##  ¿Cuál componente está dominando?

In [834]:
df[["aporte_topsis", "aporte_ipc", "aporte_trend"]].mean()

aporte_topsis    0.311664
aporte_ipc       0.173567
aporte_trend     0.037930
dtype: float64

In [835]:
def generate_explanations(df):
    explanations = []

    for country, row in df.iterrows():
        # Componentes
        t = row["aporte_topsis"]
        i = row["aporte_ipc"]
        tr = row["aporte_trend"]
        total = row["radar_score"]

        # Participación porcentual
        t_pct = t / total
        i_pct = i / total
        tr_pct = tr / total

        # Ranking change
        delta = row["delta_rank"]

        # Identificar driver principal
        components = {
            "TOPSIS": t_pct,
            "IPC": i_pct,
            "Tendencia": tr_pct,
        }

        main_driver = max(components, key=components.get)
        
        # Segundo driver
        sorted_comp = sorted(components.items(), key=lambda x: x[1], reverse=True)
        second_driver = sorted_comp[1][0]

        # Etiquetas de intensidad
        def label(p):
            if p > 0.5:
                return "dominante"
            elif p > 0.3:
                return "fuerte"
            elif p > 0.2:
                return "moderado"
            else:
                return "bajo"

        # Construcción del mensaje
        if delta > 0:
            movement = f"sube {delta} posiciones"
        elif delta < 0:
            movement = f"pierde {abs(delta)} posiciones"
        else:
            movement = "mantiene su posición"

        explanation = (
            f"{country} {movement} en el ranking RADAR. "
            f"El principal driver es {main_driver} ({components[main_driver]:.0%}, impacto {label(components[main_driver])}), "
            f"seguido por {second_driver}. "
            f"La contribución de TOPSIS es {t_pct:.0%}, IPC {i_pct:.0%} y Tendencia {tr_pct:.0%}."
        )

        explanations.append(explanation)

    df["explain"] = explanations
    return df

In [836]:
df_explain = generate_explanations(df)

df_explain[["radar_score", "rank_radar", "rank_topsis", "delta_rank", "explain"]]#.head(10)

,radar_score,rank_radar,rank_topsis,delta_rank,explain
country_iso3,,,,,
PAN,0.722897,1,8,7,PAN sube 7.0 posiciones en el ranking RADAR. E...
CRI,0.659813,2,6,4,CRI sube 4.0 posiciones en el ranking RADAR. E...
DOM,0.633875,3,9,6,DOM sube 6.0 posiciones en el ranking RADAR. E...
ESP,0.626675,4,3,-1,ESP pierde 1.0 posiciones en el ranking RADAR....
CHL,0.591509,5,4,-1,CHL pierde 1.0 posiciones en el ranking RADAR....
ECU,0.582632,6,22,16,ECU sube 16.0 posiciones en el ranking RADAR. ...
PER,0.559473,7,17,10,PER sube 10.0 posiciones en el ranking RADAR. ...
MEX,0.558872,8,12,4,MEX sube 4.0 posiciones en el ranking RADAR. E...
GTM,0.548403,9,19,10,GTM sube 10.0 posiciones en el ranking RADAR. ...


In [837]:
check = df[["radar_score"]].merge(
    results["radar_global"][["country_iso3", "radar_score"]].set_index("country_iso3"),
    left_index=True,
    right_index=True,
    suffixes=("_composite", "_global"),
)

check["diff"] = check["radar_score_composite"] - check["radar_score_global"]

check.sort_values("diff", ascending=False)


,radar_score_composite,radar_score_global,diff
country_iso3,,,
PAN,0.722897,0.722897,0.0
CRI,0.659813,0.659813,0.0
SUR,0.403891,0.403891,0.0
BLZ,0.425406,0.425406,0.0
TTO,0.435269,0.435269,0.0
BRA,0.438470,0.438470,0.0
JAM,0.455292,0.455292,0.0
GUY,0.479533,0.479533,0.0
BRB,0.489188,0.489188,0.0


In [838]:
results["decision_matrix"]#.head(10).round(3)

variable,gdp_nominal,gdp_per_capita_ppp,inflation_rate,population_total,urban_population_pct,unemployment_rate,current_account_gdp,public_debt_gdp,trade_openness,fdi_net_inflows_gdp,...,control_of_corruption,country_risk_premium,heritage_efi,internet_users_pct,mobile_subscriptions,digital_payment_adults_pct,secure_internet_servers_per_million,commercial_bank_branches_per_100k_adults,atms_per_100k_adults,ict_goods_exports_pct_total_goods_exports
country_iso3,,,,,,,,,,,,,,,,,,,,,
ARG,0.581690,0.329687,0.137314,0.717027,0.951911,0.632010,0.366979,0.515770,0.033014,0.328804,...,0.391865,0.635858,0.623188,0.872843,0.674096,0.681100,0.578476,0.456250,0.658154,0.001684
BHS,0.175541,0.460010,0.996781,0.049497,0.793502,0.464911,0.000000,0.479288,0.331166,0.292636,...,0.875796,0.781401,0.782609,0.931419,0.282865,0.423450,0.593809,0.640412,0.763867,0.039843
BLZ,0.000000,0.134996,0.985501,0.054937,0.220998,0.493355,0.245046,0.572398,0.503911,0.510352,...,0.443214,0.635858,0.774327,0.671129,0.017840,0.424201,1.000000,0.512201,0.542293,0.007856
BOL,0.312112,0.117216,0.978412,0.533302,0.647542,0.970583,0.207814,0.507131,0.143693,0.159006,...,0.183146,0.417259,0.312629,0.664842,0.294982,0.332414,0.307436,1.000000,0.512925,0.001684
BRA,0.716915,0.231732,0.981280,0.933362,0.888566,0.727229,0.176219,0.386440,0.077492,0.468489,...,0.366024,0.878429,0.519669,0.764198,0.329996,0.743942,0.602660,0.540167,0.752254,0.015152
BRB,0.093424,0.261799,0.992719,0.000000,0.478324,0.682334,0.069742,0.000000,0.171616,0.512364,...,0.873833,0.732887,0.892340,0.469836,0.443658,0.423450,0.424357,0.476255,0.444719,0.061728
CAN,0.719783,0.743398,0.989057,0.702728,0.813399,0.651297,0.299501,0.537962,0.249281,0.423664,...,1.000000,1.000000,1.000000,0.970683,0.259745,1.000000,0.777542,0.596512,0.878013,0.076880
CHL,0.509289,0.399286,0.981554,0.598874,0.904470,0.483793,0.252023,1.000000,0.241785,0.496220,...,0.836563,0.958906,0.973085,0.996509,0.606058,0.828429,0.656813,0.370697,0.552881,0.016274
CRI,0.372799,0.337865,1.000000,0.408725,0.764362,0.656483,0.277466,0.766135,0.285196,0.595573,...,0.704052,0.853887,0.865424,0.820711,0.636275,0.537679,0.477917,0.536153,0.618757,0.046577


In [839]:
results["wide_raw"]#.head(10).round(3)

variable,gdp_nominal,gdp_per_capita_ppp,gdp_growth,inflation_rate,population_total,urban_population_pct,unemployment_rate,current_account_gdp,public_debt_gdp,trade_openness,...,geographic_distance_km,common_language_spanish,hofstede_pdi,hofstede_idv,hofstede_mas,hofstede_uai,hofstede_lto,hofstede_ivr,colombian_diaspora_stock,cultural_distance_hofstede
country_iso3,,,,,,,,,,,,,,,,,,,,,
ARG,27.182177,30431.193122,-1.342931,219.883929,17.637525,92.274232,7.145,0.893118,67.372401,27.929761,...,8.457655,1.0,49.0,51.0,56.0,86.0,29.0,62.0,9.486987,43.335897
BHS,23.485350,41197.934255,3.378666,0.409162,12.902425,81.324926,9.207,-6.648608,71.457821,79.242459,...,7.779467,0.0,47.0,38.0,65.0,45.0,14.0,67.0,5.926926,45.022217
BLZ,21.887551,14346.518805,3.504664,3.289560,12.941017,41.753211,8.856,-1.612700,61.030832,108.972447,...,7.754053,0.0,80.0,19.0,40.0,86.0,22.5,89.0,7.598650,34.485504
BOL,24.728439,12877.635118,-1.123356,5.099766,16.334280,71.236145,2.967,-2.377848,68.339853,46.977872,...,7.501634,1.0,78.0,23.0,42.0,87.0,21.0,46.0,9.238199,47.791213
BRA,28.413013,22338.476564,3.419315,4.367464,19.172090,87.895855,5.970,-3.027160,81.855443,35.584524,...,8.368925,0.0,69.0,36.0,49.0,76.0,28.0,59.0,9.173054,36.796739
BRB,22.737909,24822.534331,2.482262,1.446437,12.551321,59.539703,6.524,-5.215344,125.131136,51.783514,...,7.528869,0.0,47.0,38.0,65.0,45.0,14.0,67.0,3.433987,45.022217
CAN,28.439119,64610.379517,1.554795,2.381584,17.536097,82.700257,6.907,-0.493602,64.887158,65.149921,...,8.571871,0.0,39.0,72.0,52.0,48.0,54.0,68.0,9.143346,79.561297
CHL,26.523168,36181.156617,2.644312,4.297639,16.799412,88.995091,8.974,-1.469331,13.145640,63.859857,...,8.354910,1.0,63.0,49.0,28.0,86.0,12.0,68.0,9.299358,44.821870
CRI,25.280825,31106.764356,4.321224,-0.412853,15.450599,79.310767,6.843,-0.946456,39.335112,71.331014,...,7.074117,1.0,35.0,15.0,21.0,86.0,22.5,89.0,8.503094,58.423026


In [840]:
results["wide_raw"].columns

Index(['gdp_nominal', 'gdp_per_capita_ppp', 'gdp_growth', 'inflation_rate',
       'population_total', 'urban_population_pct', 'unemployment_rate',
       'current_account_gdp', 'public_debt_gdp', 'trade_openness',
       'fdi_net_inflows_gdp', 'weighted_mean_applied_tariff_all_products',
       'domestic_credit_private_gdp', 'account_ownership',
       'interest_rate_spread', 'bank_npl_ratio', 'stock_market_cap_gdp',
       'personal_remittances_gdp', 'bank_concentration_5',
       'financial_system_deposits_gdp', 'bank_capital_rwa',
       'regulatory_quality', 'government_effectiveness', 'rule_of_law',
       'political_stability', 'voice_accountability', 'control_of_corruption',
       'country_risk_premium', 'heritage_efi', 'internet_users_pct',
       'mobile_subscriptions', 'digital_payment_adults_pct',
       'secure_internet_servers_per_million',
       'commercial_bank_branches_per_100k_adults', 'atms_per_100k_adults',
       'ict_goods_exports_pct_total_goods_exports', 'geog

In [841]:
from typing import Any, Dict

import pandas as pd

from src.scoring.hybrid_scorer import _build_business_line_weights



def audit_business_line_weights(
    configs: Dict[str, Dict[str, Any]],
    decision_matrix: pd.DataFrame,
) -> pd.DataFrame:
    """Audita los pesos efectivos usados por TOPSIS para cada línea de negocio.

    Permite verificar:
    - peso global por dimensión;
    - peso de dimensión por línea;
    - peso global de variable dentro de dimensión;
    - override por variable, si existe;
    - peso final TOPSIS usado después de filtrar por decision_matrix.
    """

    business_lines = configs["business_lines"]["business_lines"]
    global_dim_weights = configs["weights"]["dimension_weights"]
    global_variable_weights = configs["weights"]["variable_weights"]

    rows = []

    for bl_key, bl_cfg in business_lines.items():
        dim_weights_line, final_var_weights = _build_business_line_weights(
            business_line_cfg=bl_cfg,
            variable_weights_by_dim=global_variable_weights,
        )

        # Replicar la lógica usada antes de TOPSIS:
        # 1. filtrar variables presentes en decision_matrix;
        # 2. renormalizar pesos restantes para que sumen 1.
        final_var_weights_filtered = {
            var: weight
            for var, weight in final_var_weights.items()
            if var in decision_matrix.columns
        }

        total_filtered = sum(final_var_weights_filtered.values())

        if total_filtered > 0:
            final_var_weights_filtered = {
                var: weight / total_filtered
                for var, weight in final_var_weights_filtered.items()
            }

        overrides = bl_cfg.get("variable_weight_overrides", {}) or {}

        for dim, vars_global in global_variable_weights.items():
            dim_weight_line = dim_weights_line.get(dim, 0.0)
            dim_weight_global = global_dim_weights.get(dim)

            for var, global_var_weight in vars_global.items():
                override_weight = None

                if dim in overrides and var in overrides[dim]:
                    override_weight = overrides[dim][var]

                rows.append(
                    {
                        "business_line": bl_key,
                        "dimension": dim,
                        "variable": var,
                        "in_decision_matrix": var in decision_matrix.columns,
                        "global_dimension_weight": dim_weight_global,
                        "line_dimension_weight": dim_weight_line,
                        "global_variable_weight_in_dim": global_var_weight,
                        "override_variable_weight_in_dim": override_weight,
                        "has_override": override_weight is not None,
                        "final_topsis_weight": final_var_weights_filtered.get(var, 0.0),
                    }
                )

    audit = pd.DataFrame(rows)

    return (
        audit
        .sort_values(
            ["business_line", "dimension", "final_topsis_weight"],
            ascending=[True, True, False],
        )
        .reset_index(drop=True)
    )


In [842]:
weights_audit = audit_business_line_weights(
    configs=configs,
    decision_matrix=decision_matrix,
)

weights_audit#.head(30)

,business_line,dimension,variable,in_decision_matrix,global_dimension_weight,line_dimension_weight,global_variable_weight_in_dim,override_variable_weight_in_dim,has_override,final_topsis_weight
0,AD,digital_tech,secure_internet_servers_per_million,True,0.1,0.37,0.15,0.25,True,0.092500
1,AD,digital_tech,digital_payment_adults_pct,True,0.1,0.37,0.21,0.23,True,0.085100
2,AD,digital_tech,internet_users_pct,True,0.1,0.37,0.20,0.20,True,0.074000
3,AD,digital_tech,ict_goods_exports_pct_total_goods_exports,True,0.1,0.37,0.10,0.16,True,0.059200
4,AD,digital_tech,mobile_subscriptions,True,0.1,0.37,0.19,0.10,True,0.037000
...,...,...,...,...,...,...,...,...,...,...
170,PF,macro,current_account_gdp,True,0.3,0.18,0.07,0.07,True,0.012727
171,PF,macro,gdp_nominal,True,0.3,0.18,0.17,0.06,True,0.010909
172,PF,macro,fdi_net_inflows_gdp,True,0.3,0.18,0.06,0.06,True,0.010909
173,PF,macro,population_total,True,0.3,0.18,0.10,0.04,True,0.007273


In [843]:
weights_audit[
    ~weights_audit["in_decision_matrix"]
][
    ["business_line", "dimension", "variable", "global_variable_weight_in_dim"]
].drop_duplicates()

,business_line,dimension,variable,global_variable_weight_in_dim


In [844]:
weights_audit[
    ~weights_audit["in_decision_matrix"]
]["variable"].unique()

array([], dtype=object)

In [845]:
from src.utils import get_variable_catalog, get_world_bank_variable_catalog

var = "digital_payment_adults_pct"

catalog = get_variable_catalog(configs["variables"])

wb_catalog = get_world_bank_variable_catalog(
    configs["variables"],
    configs["data_sources"]["sources"]["world_bank"],
)

print("1. En catalog general:", var in catalog)
print("2. En catalog WB:", var in wb_catalog)
print("3. En weights.yaml:",
      any(var in dim_vars for dim_vars in configs["weights"]["variable_weights"].values()))
print("4. En master:",
      var in master["variable"].astype(str).str.strip().unique())
print("5. En wide_raw:",
      var in wide_raw.columns)
print("6. En decision_matrix:",
      var in decision_matrix.columns)

if var in wb_catalog:
    print("\nMetadata WB:")
    print(wb_catalog[var])

1. En catalog general: True
2. En catalog WB: True
3. En weights.yaml: True
4. En master: True
5. En wide_raw: True
6. En decision_matrix: True

Metadata WB:
{'source': 'world_bank', 'db': 28, 'indicator_code': 'g20.any', 'direction': 'positive', 'frequency': 'triennial', 'description': 'Adultos que hicieron o recibieron pagos digitales en el ultimo ano', 'name': 'digital_payment_adults_pct', 'dimension': 'digital_tech'}


In [846]:
master["variable"].unique()

array(['account_ownership', 'atms_per_100k_adults', 'bank_capital_rwa',
       'bank_concentration_5', 'bank_npl_ratio',
       'colombian_diaspora_stock',
       'commercial_bank_branches_per_100k_adults',
       'common_language_spanish', 'control_of_corruption',
       'country_risk_premium', 'current_account_gdp',
       'digital_payment_adults_pct', 'domestic_credit_private_gdp',
       'fdi_net_inflows_gdp', 'financial_system_deposits_gdp',
       'gdp_growth', 'gdp_nominal', 'gdp_per_capita_ppp',
       'geographic_distance_km', 'government_effectiveness',
       'heritage_efi', 'hofstede_idv', 'hofstede_ivr', 'hofstede_lto',
       'hofstede_mas', 'hofstede_pdi', 'hofstede_uai',
       'ict_goods_exports_pct_total_goods_exports', 'inflation_rate',
       'interest_rate_spread', 'internet_users_pct',
       'mobile_subscriptions', 'personal_remittances_gdp',
       'political_stability', 'population_total', 'regulatory_quality',
       'rule_of_law', 'secure_internet_servers_per

In [847]:
from src.utils import get_world_bank_variable_catalog

wb_catalog = get_world_bank_variable_catalog(
    configs["variables"],
    configs["data_sources"]["sources"]["world_bank"],
)

"digital_payment_adults_pct" in wb_catalog

True

In [848]:
from src.utils import get_variable_catalog, get_world_bank_variable_catalog

catalog = get_variable_catalog(configs["variables"])

print("En catalog general:", "digital_payment_adults_pct" in catalog)

wb_catalog = get_world_bank_variable_catalog(
    configs["variables"],
    configs["data_sources"]["sources"]["world_bank"],
)

print("En catalog WB:", "digital_payment_adults_pct" in wb_catalog)
print(wb_catalog.get("digital_payment_adults_pct"))

En catalog general: True
En catalog WB: True
{'source': 'world_bank', 'db': 28, 'indicator_code': 'g20.any', 'direction': 'positive', 'frequency': 'triennial', 'description': 'Adultos que hicieron o recibieron pagos digitales en el ultimo ano', 'name': 'digital_payment_adults_pct', 'dimension': 'digital_tech'}


In [849]:
#Validar que los pesos finales sumen 1 por línea
(
    weights_audit[weights_audit["in_decision_matrix"]]
    .groupby("business_line")["final_topsis_weight"]
    .sum()
    .round(6)
)

business_line
AD     1.0
BD     1.0
CIB    1.0
IB     1.0
PF     1.0
Name: final_topsis_weight, dtype: float64

In [850]:
#Esto te dirá qué variables realmente diferencian líneas.
weight_matrix = (
    weights_audit[weights_audit["in_decision_matrix"]]
    .pivot_table(
        index="variable",
        columns="business_line",
        values="final_topsis_weight",
        aggfunc="sum",
        fill_value=0.0,
    )
)

weight_matrix["spread"] = (
    weight_matrix.max(axis=1) - weight_matrix.min(axis=1)
)

weight_matrix.sort_values("spread", ascending=False)#.head(20)

business_line,AD,BD,CIB,IB,PF,spread
variable,,,,,,
digital_payment_adults_pct,0.0851,0.0805,0.0036,0.0128,0.134400,0.130800
stock_market_cap_gdp,0.0260,0.0023,0.1140,0.0129,0.008526,0.111700
internet_users_pct,0.0740,0.1015,0.0054,0.0176,0.067200,0.096100
mobile_subscriptions,0.0370,0.0665,0.0030,0.0144,0.092400,0.089400
secure_internet_servers_per_million,0.0925,0.0595,0.0084,0.0176,0.029400,0.084100
domestic_credit_private_gdp,0.0182,0.0345,0.0646,0.0946,0.028421,0.076400
personal_remittances_gdp,0.0052,0.0046,0.0076,0.0043,0.079579,0.075279
regulatory_quality,0.1026,0.0374,0.0594,0.0540,0.028600,0.074000
financial_system_deposits_gdp,0.0156,0.0253,0.0570,0.0860,0.028421,0.070400


In [851]:
for bl in weights_audit["business_line"].unique():
    print(f"\n--- {bl} ---")
    display(
        weights_audit[
            (weights_audit["business_line"] == bl)
            & (weights_audit["in_decision_matrix"])
        ][
            [
                "dimension",
                "variable",
                "has_override",
                "line_dimension_weight",
                "global_variable_weight_in_dim",
                "override_variable_weight_in_dim",
                "final_topsis_weight",
            ]
        ]
        .sort_values("final_topsis_weight", ascending=False)
        #.head(15)
    )


--- AD ---


,dimension,variable,has_override,line_dimension_weight,global_variable_weight_in_dim,override_variable_weight_in_dim,final_topsis_weight
16,institutional,regulatory_quality,True,0.38,0.13,0.27,0.1026
0,digital_tech,secure_internet_servers_per_million,True,0.37,0.15,0.25,0.0925
1,digital_tech,digital_payment_adults_pct,True,0.37,0.21,0.23,0.0851
17,institutional,rule_of_law,True,0.38,0.13,0.22,0.0836
2,digital_tech,internet_users_pct,True,0.37,0.20,0.20,0.0740
18,institutional,control_of_corruption,True,0.38,0.13,0.17,0.0646
3,digital_tech,ict_goods_exports_pct_total_goods_exports,True,0.37,0.10,0.16,0.0592
19,institutional,government_effectiveness,True,0.38,0.13,0.13,0.0494
20,institutional,political_stability,True,0.38,0.12,0.10,0.0380
4,digital_tech,mobile_subscriptions,True,0.37,0.19,0.10,0.0370



--- BD ---


,dimension,variable,has_override,line_dimension_weight,global_variable_weight_in_dim,override_variable_weight_in_dim,final_topsis_weight
35,digital_tech,internet_users_pct,True,0.35,0.20,0.29,0.1015
36,digital_tech,digital_payment_adults_pct,True,0.35,0.21,0.23,0.0805
37,digital_tech,mobile_subscriptions,True,0.35,0.19,0.19,0.0665
42,financial,account_ownership,True,0.23,0.13,0.28,0.0644
38,digital_tech,secure_internet_servers_per_million,True,0.35,0.15,0.17,0.0595
59,macro,population_total,True,0.25,0.10,0.22,0.0550
60,macro,gdp_per_capita_ppp,True,0.25,0.17,0.18,0.0450
43,financial,bank_concentration_5,True,0.23,0.10,0.17,0.0391
51,institutional,regulatory_quality,True,0.17,0.13,0.22,0.0374
61,macro,urban_population_pct,True,0.25,0.08,0.14,0.0350



--- CIB ---


,dimension,variable,has_override,line_dimension_weight,global_variable_weight_in_dim,override_variable_weight_in_dim,final_topsis_weight
77,financial,stock_market_cap_gdp,True,0.38,0.10,0.30,0.1140
86,institutional,rule_of_law,True,0.27,0.13,0.25,0.0675
94,macro,gdp_nominal,True,0.32,0.17,0.21,0.0672
78,financial,domestic_credit_private_gdp,True,0.38,0.13,0.17,0.0646
87,institutional,regulatory_quality,True,0.27,0.13,0.22,0.0594
79,financial,financial_system_deposits_gdp,True,0.38,0.12,0.15,0.0570
80,financial,bank_capital_rwa,True,0.38,0.12,0.13,0.0494
88,institutional,government_effectiveness,True,0.27,0.13,0.18,0.0486
96,macro,trade_openness,True,0.32,0.07,0.14,0.0448
95,macro,gdp_per_capita_ppp,True,0.32,0.17,0.14,0.0448



--- IB ---


,dimension,variable,has_override,line_dimension_weight,global_variable_weight_in_dim,override_variable_weight_in_dim,final_topsis_weight
112,financial,domestic_credit_private_gdp,True,0.43,0.13,0.22,0.0946
113,financial,financial_system_deposits_gdp,True,0.43,0.12,0.20,0.0860
114,financial,bank_npl_ratio,True,0.43,0.10,0.14,0.0602
121,institutional,rule_of_law,True,0.27,0.13,0.22,0.0594
122,institutional,regulatory_quality,True,0.27,0.13,0.20,0.0540
115,financial,account_ownership,True,0.43,0.13,0.12,0.0516
116,financial,bank_capital_rwa,True,0.43,0.12,0.12,0.0516
123,institutional,government_effectiveness,True,0.27,0.13,0.17,0.0459
124,institutional,control_of_corruption,True,0.27,0.13,0.15,0.0405
129,macro,gdp_per_capita_ppp,True,0.22,0.17,0.16,0.0352



--- PF ---


,dimension,variable,has_override,line_dimension_weight,global_variable_weight_in_dim,override_variable_weight_in_dim,final_topsis_weight
140,digital_tech,digital_payment_adults_pct,True,0.42,0.21,0.32,0.134400
141,digital_tech,mobile_subscriptions,True,0.42,0.19,0.22,0.092400
147,financial,personal_remittances_gdp,True,0.27,0.10,0.28,0.079579
142,digital_tech,internet_users_pct,True,0.42,0.20,0.16,0.067200
148,financial,account_ownership,True,0.27,0.13,0.22,0.062526
143,digital_tech,atms_per_100k_adults,True,0.42,0.05,0.11,0.046200
164,macro,trade_openness,True,0.18,0.07,0.22,0.040000
144,digital_tech,commercial_bank_branches_per_100k_adults,True,0.42,0.10,0.08,0.033600
145,digital_tech,secure_internet_servers_per_million,True,0.42,0.15,0.07,0.029400
165,macro,inflation_rate,True,0.18,0.08,0.16,0.029091


In [852]:
configs["business_lines"]["business_lines"]["AD"].keys()

dict_keys(['label', 'description', 'weight_profile', 'variable_weight_overrides', 'key_variables', 'signal_logic'])

In [853]:
configs["business_lines"]["business_lines"]["AD"].get("variable_weight_overrides")

{'macro': {'gdp_nominal': 0.06,
  'gdp_per_capita_ppp': 0.21,
  'inflation_rate': 0.2,
  'population_total': 0.04,
  'urban_population_pct': 0.04,
  'unemployment_rate': 0.05,
  'current_account_gdp': 0.05,
  'public_debt_gdp': 0.15,
  'trade_openness': 0.08,
  'fdi_net_inflows_gdp': 0.08,
  'weighted_mean_applied_tariff_all_products': 0.04},
 'financial': {'stock_market_cap_gdp': 0.2,
  'domestic_credit_private_gdp': 0.14,
  'financial_system_deposits_gdp': 0.12,
  'account_ownership': 0.11,
  'bank_capital_rwa': 0.13,
  'bank_npl_ratio': 0.12,
  'interest_rate_spread': 0.08,
  'bank_concentration_5': 0.06,
  'personal_remittances_gdp': 0.04},
 'institutional': {'regulatory_quality': 0.27,
  'rule_of_law': 0.22,
  'control_of_corruption': 0.17,
  'government_effectiveness': 0.13,
  'political_stability': 0.1,
  'country_risk_premium': 0.06,
  'heritage_efi': 0.03,
  'voice_accountability': 0.02},
 'digital_tech': {'secure_internet_servers_per_million': 0.25,
  'digital_payment_adults_

In [854]:
rankings = {}

for bl, df_bl in results["business_line_rankings"].items():
    tmp = df_bl.copy()

    if "country_iso3" in tmp.columns:
        tmp = tmp.set_index("country_iso3")

    rankings[bl] = tmp["rank"]

rank_df = pd.DataFrame(rankings)

rank_df.corr(method="spearman")

,IB,PF,AD,BD,CIB
IB,1.000000,0.646962,0.844007,0.684729,0.823755
PF,0.646962,1.000000,0.813355,0.895457,0.610837
AD,0.844007,0.813355,1.000000,0.872469,0.818829
BD,0.684729,0.895457,0.872469,1.000000,0.667214
CIB,0.823755,0.610837,0.818829,0.667214,1.000000


In [855]:
#validación: gaps de score

# creterio:
# gap < 0.005       empate práctico
# 0.005 - 0.015     diferencia débil
# 0.015 - 0.030     diferencia moderada
# > 0.030           diferencia material

# Si varios países tienen gaps pequeños, el output ejecutivo debe presentarse por bandas de atractividad, no solo ranking ordinal.

def assign_tier_by_score(score: float, q80: float, q60: float, q40: float) -> str:
    if score >= q80:
        return "Alta"
    if score >= q60:
        return "Media-alta"
    if score >= q40:
        return "Media"
    return "Baja"


tier_tables = {}

for bl, df_bl in results["business_line_rankings"].items():
    tmp = df_bl.copy()

    if "country_iso3" in tmp.columns:
        tmp = tmp.set_index("country_iso3")

    tmp = tmp.sort_values("score", ascending=False).copy()

    q80 = tmp["score"].quantile(0.80)
    q60 = tmp["score"].quantile(0.60)
    q40 = tmp["score"].quantile(0.40)

    tmp["attractiveness_tier"] = tmp["score"].apply(
        lambda x: assign_tier_by_score(x, q80, q60, q40)
    )

    tmp["score_gap_next"] = tmp["score"] - tmp["score"].shift(-1)
    tmp["gap_interpretation"] = tmp["score_gap_next"].apply(classify_gap)

    tier_tables[bl] = tmp[
        ["score", "rank", "attractiveness_tier", "score_gap_next", "gap_interpretation"]
    ]



In [856]:
def strategic_read(row):
    tier = row["attractiveness_tier"]
    gap = row["gap_interpretation"]

    if tier == "Alta" and gap == "Diferencia material":
        return "Liderazgo o posicionamiento claramente diferenciado"
    if tier == "Alta" and gap in ["Empate práctico", "Diferencia débil"]:
        return "Alta atractividad, pero no distinguible ordinalmente del país siguiente"
    if tier == "Media-alta" and gap == "Empate práctico":
        return "Banda competitiva media-alta; decisión requiere análisis de drivers"
    if tier == "Media":
        return "Atractividad intermedia; priorizar solo si hay racional estratégico específico"
    return "Lectura dependiente de drivers y restricciones de implementación"


for bl, table in tier_tables.items():
    table = table.copy()
    table["strategic_read"] = table.apply(strategic_read, axis=1)
    tier_tables[bl] = table

In [857]:
tier_tables["PF"]#.head(15).round(3)

,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
country_iso3,,,,,,
ESP,0.668780,1,Alta,0.036149,Diferencia material,Liderazgo o posicionamiento claramente diferen...
USA,0.632632,2,Alta,0.005514,Diferencia débil,"Alta atractividad, pero no distinguible ordina..."
CAN,0.627117,3,Alta,0.012880,Diferencia débil,"Alta atractividad, pero no distinguible ordina..."
CHL,0.614237,4,Alta,0.033931,Diferencia material,Liderazgo o posicionamiento claramente diferen...
URY,0.580306,5,Alta,0.019760,Diferencia moderada,Lectura dependiente de drivers y restricciones...
SUR,0.560546,6,Alta,0.012751,Diferencia débil,"Alta atractividad, pero no distinguible ordina..."
ARG,0.547795,7,Media-alta,0.001263,Empate práctico,Banda competitiva media-alta; decisión requier...
CRI,0.546532,8,Media-alta,0.004164,Empate práctico,Banda competitiva media-alta; decisión requier...
BRA,0.542368,9,Media-alta,0.003683,Empate práctico,Banda competitiva media-alta; decisión requier...


El modelo ya no debe interpretarse como cinco rankings independientes, sino como cinco tesis de atractividad. Las líneas muestran diferencias relevantes: CIB e IB privilegian profundidad financiera, escala e institucionalidad; PF privilegia transaccionalidad y flujos; BD privilegia escala retail digital; AD privilegia madurez digital e institucional. Sin embargo, dentro de cada línea existen bandas donde las diferencias de score son pequeñas y no deben sobrerrepresentarse como posiciones ordinales fuertes.

In [858]:
#dispersión de ranking entre líneas

# qué países cambian realmente de atractividad según la línea:
# Países con rank_range_across_lines alto son los más sensibles a la tesis de negocio. Esos son estratégicamente interesantes porque muestran donde la línea de negocio cambia la decisión.

rank_cols = ["IB", "PF", "AD", "BD", "CIB"]

rank_df_clean = rank_df[rank_cols].copy()

rank_df_clean["rank_std_across_lines"] = rank_df_clean[rank_cols].std(axis=1)
rank_df_clean["rank_range_across_lines"] = (
    rank_df_clean[rank_cols].max(axis=1)
    - rank_df_clean[rank_cols].min(axis=1)
)

rank_df_clean.sort_values(
    "rank_range_across_lines",
    ascending=False
)#.head(15)

,IB,PF,AD,BD,CIB,rank_std_across_lines,rank_range_across_lines
country_iso3,,,,,,,
ARG,25,7,10,6,21,8.642916,19
SUR,19,6,16,12,23,6.534524,17
BRB,9,20,8,23,6,7.726578,17
GUY,24,11,18,18,16,4.669047,13
BHS,5,18,7,9,5,5.403702,13
MEX,23,22,19,15,12,4.658326,11
BLZ,13,21,11,19,17,4.147288,10
URY,6,5,5,5,15,4.381780,10
SLV,14,16,21,24,18,3.974921,10


### contribuciones para todas las líneas

Importante: esto no es la descomposición matemática exacta del score TOPSIS, porque TOPSIS calcula distancia al ideal positivo/negativo. Pero sí es una explicabilidad ejecutiva robusta.

NO es “contribución exacta al TOPSIS” es Contribución ponderada post-normalización

In [859]:
contrib_by_line = compute_all_business_line_contributions(
    decision_matrix=decision_matrix,
    weights_audit=weights_audit,
)
if isinstance(contrib_by_line, dict):
    try:
        contrib_df = pd.concat(contrib_by_line, axis=1)
    except Exception:
        contrib_df = pd.DataFrame.from_dict(contrib_by_line)
else:
    contrib_df = pd.DataFrame(contrib_by_line)

contrib_df

AD                              \
     business_line country_iso3      dimension   
812             AD          ARG   digital_tech   
870             AD          ARG   digital_tech   
899             AD          ARG   digital_tech   
580             AD          ARG  institutional   
638             AD          ARG  institutional   
...            ...          ...            ...   
724             AD          VEN  institutional   
753             AD          VEN  institutional   
782             AD          VEN  institutional   
811             AD          VEN  institutional   
1014            AD          VEN   digital_tech   

                                                                  \
                                       variable normalized_value   
812                          internet_users_pct         0.872843   
870                  digital_payment_adults_pct         0.681100   
899         secure_internet_servers_per_million         0.578476   
580                          regulatory_quality         0.510843   
638                                 rule_of_law         0.532835   
...                                         ...              ...   
724                        voice_accountability         0.000000   
753                       control_of_corruption         0.000000   
782                        country_risk_premium         0.000000   
811                                heritage_efi         0.000000   
1014  ict_goods_exports_pct_total_goods_exports         0.000000   

                                                              \
     final_topsis_weight contribution shortfall has_override   
812               0.0740     0.064590  0.009410         True   
870               0.0851     0.057962  0.027138         True   
899               0.0925     0.053509  0.038991         True   
580               0.1026     0.052412  0.050188         True   
638               0.0836     0.044545  0.039055         True   
...                  ...          ...       ...          ...   
724               0.0076     0.000000  0.007600         True   
753               0.0646     0.000000  0.064600         True   
782               0.0228     0.000000  0.022800         True   
811               0.0114     0.000000  0.011400         True   
1014              0.0592     0.000000  0.059200         True   

                                      ...            PF               \
     override_variable_weight_in_dim  ... business_line country_iso3   
812                             0.20  ...            PF          ARG   
870                             0.23  ...            PF          ARG   
899                             0.25  ...            PF          ARG   
580                             0.27  ...            PF          ARG   
638                             0.22  ...            PF          ARG   
...                              ...  ...           ...          ...   
724                             0.02  ...            PF          VEN   
753                             0.17  ...            PF          VEN   
782                             0.06  ...            PF          VEN   
811                             0.03  ...            PF          VEN   
1014                            0.16  ...            PF          VEN   

                                                                \
          dimension                                   variable   
812    digital_tech                         internet_users_pct   
870    digital_tech                 digital_payment_adults_pct   
899    digital_tech        secure_internet_servers_per_million   
580   institutional                         regulatory_quality   
638   institutional                                rule_of_law   
...             ...                                        ...   
724   institutional                       voice_accountability   
753   institutional                      control_of_corruption   
782   institutional                       country_

### principales drivers positivos por país y línea

In [860]:
get_top_contributors(
    contributions=contrib_by_line["PF"],
    country_iso3="ARG",
    top_n=8,
)

,business_line,country_iso3,dimension,variable,normalized_value,final_topsis_weight,contribution
870,PF,ARG,digital_tech,digital_payment_adults_pct,0.681100,0.134400,0.091540
841,PF,ARG,digital_tech,mobile_subscriptions,0.674096,0.092400,0.062286
812,PF,ARG,digital_tech,internet_users_pct,0.872843,0.067200,0.058655
348,PF,ARG,financial,account_ownership,0.777665,0.062526,0.048625
957,PF,ARG,digital_tech,atms_per_100k_adults,0.658154,0.046200,0.030407
899,PF,ARG,digital_tech,secure_internet_servers_per_million,0.578476,0.029400,0.017007
377,PF,ARG,financial,interest_rate_spread,0.807242,0.019895,0.016060
928,PF,ARG,digital_tech,commercial_bank_branches_per_100k_adults,0.456250,0.033600,0.015330


### Principales restricciones o brechas

In [861]:
get_top_shortfalls(
    contributions=contrib_by_line["PF"],
    country_iso3="ARG",
    top_n=8,
)

,business_line,country_iso3,dimension,variable,normalized_value,final_topsis_weight,shortfall
464,PF,ARG,financial,personal_remittances_gdp,0.036989,0.079579,0.076635
870,PF,ARG,digital_tech,digital_payment_adults_pct,0.681100,0.134400,0.042860
232,PF,ARG,macro,trade_openness,0.033014,0.040000,0.038679
841,PF,ARG,digital_tech,mobile_subscriptions,0.674096,0.092400,0.030114
522,PF,ARG,financial,financial_system_deposits_gdp,0.000000,0.028421,0.028421
58,PF,ARG,macro,inflation_rate,0.137314,0.029091,0.025096
319,PF,ARG,financial,domestic_credit_private_gdp,0.323269,0.028421,0.019233
928,PF,ARG,digital_tech,commercial_bank_branches_per_100k_adults,0.456250,0.033600,0.018270


### Resumen por dimensión

Esto es útil para explicar de forma ejecutiva:

Ejemplo: gentina en PF destaca por digital_tech y financial, pero se limita por institutional/macro.

In [862]:
dimension_summary_pf = summarize_contributions_by_dimension(
    contrib_by_line["PF"]
)

dimension_summary_pf[
    dimension_summary_pf["country_iso3"] == "ARG"
]

,business_line,country_iso3,dimension,contribution,shortfall
0,PF,ARG,digital_tech,0.275254,0.144746
1,PF,ARG,financial,0.104048,0.165952
2,PF,ARG,institutional,0.069741,0.060259
3,PF,ARG,macro,0.063450,0.116550


In [863]:
print(
    generate_country_line_explanation(
        contributions=contrib_by_line["PF"],
        country_iso3="ARG",
        top_n=3,
    )
)

ARG en PF está impulsado principalmente por digital_payment_adults_pct (0.092), mobile_subscriptions (0.062), internet_users_pct (0.059). Sus principales restricciones relativas son personal_remittances_gdp (0.077), digital_payment_adults_pct (0.043), trade_openness (0.039).


### Comparar un país entre líneas

Esto es muy útil para casos como Argentina, Surinam, Uruguay o Bahamas. Esto te permite ver por qué Argentina sube en PF y BD, pero cae en IB y CIB.

In [864]:
compare_country_across_lines(
    contrib_by_line=contrib_by_line,
    country_iso3="ARG",
    top_n=5,
)

,business_line,country_iso3,dimension,variable,normalized_value,final_topsis_weight,contribution,shortfall
0,AD,ARG,digital_tech,internet_users_pct,0.872843,0.074000,0.064590,0.009410
1,AD,ARG,digital_tech,digital_payment_adults_pct,0.681100,0.085100,0.057962,0.027138
2,AD,ARG,digital_tech,secure_internet_servers_per_million,0.578476,0.092500,0.053509,0.038991
3,AD,ARG,institutional,regulatory_quality,0.510843,0.102600,0.052412,0.050188
4,AD,ARG,institutional,rule_of_law,0.532835,0.083600,0.044545,0.039055
5,BD,ARG,digital_tech,internet_users_pct,0.872843,0.101500,0.088594,0.012906
6,BD,ARG,digital_tech,digital_payment_adults_pct,0.681100,0.080500,0.054829,0.025671
7,BD,ARG,financial,account_ownership,0.777665,0.064400,0.050082,0.014318
8,BD,ARG,digital_tech,mobile_subscriptions,0.674096,0.066500,0.044827,0.021673
9,BD,ARG,macro,population_total,0.717027,0.055000,0.039436,0.015564


### Comparar países dentro de una misma línea

Ejemplo: explicar por qué España lidera PF frente a USA, CAN o CHL.

In [865]:
compare_countries_in_line(
    contributions=contrib_by_line["PF"],
    countries=["ESP", "USA", "CAN", "CHL"],
    top_n=12,
)

country_iso3,dimension,variable,CAN,CHL,ESP,USA
0,digital_tech,atms_per_100k_adults,0.040564,0.025543,0.032109,0.039633
1,digital_tech,commercial_bank_branches_per_100k_adults,0.020043,0.012455,0.026054,0.023821
2,digital_tech,digital_payment_adults_pct,0.134400,0.111341,0.133058,0.125574
3,digital_tech,internet_users_pct,0.065230,0.066965,0.067200,0.065708
4,digital_tech,mobile_subscriptions,0.024000,0.056000,0.054008,0.039840
5,digital_tech,secure_internet_servers_per_million,0.022860,0.019310,0.022032,0.027574
6,financial,account_ownership,0.062526,0.051399,0.062506,0.061373
7,financial,domestic_credit_private_gdp,0.028421,0.024026,0.023727,0.019961
8,financial,financial_system_deposits_gdp,0.028312,0.017461,0.028421,0.025575
9,institutional,regulatory_quality,0.028600,0.023507,0.022106,0.028515


Para tener una tabla ejecutiva con score, rank y drivers:

In [866]:
pf_explainability = build_explainability_table_for_line(
    ranking_df=results["business_line_rankings"]["PF"],
    contributions=contrib_by_line["PF"],
    top_n=3,
)

pf_explainability#.head(15)

,country_iso3,score,rank,top_drivers,top_constraints
0,ESP,0.668780,1,digital_payment_adults_pct; internet_users_pct...,personal_remittances_gdp; mobile_subscriptions...
1,USA,0.632632,2,digital_payment_adults_pct; internet_users_pct...,personal_remittances_gdp; mobile_subscriptions...
2,CAN,0.627117,3,digital_payment_adults_pct; internet_users_pct...,personal_remittances_gdp; mobile_subscriptions...
3,CHL,0.614237,4,digital_payment_adults_pct; internet_users_pct...,personal_remittances_gdp; mobile_subscriptions...
4,URY,0.580306,5,digital_payment_adults_pct; mobile_subscriptio...,personal_remittances_gdp; digital_payment_adul...
5,SUR,0.560546,6,digital_payment_adults_pct; mobile_subscriptio...,digital_payment_adults_pct; personal_remittanc...
6,ARG,0.547795,7,digital_payment_adults_pct; mobile_subscriptio...,personal_remittances_gdp; digital_payment_adul...
7,CRI,0.546532,8,digital_payment_adults_pct; mobile_subscriptio...,personal_remittances_gdp; digital_payment_adul...
8,BRA,0.542368,9,digital_payment_adults_pct; account_ownership;...,personal_remittances_gdp; mobile_subscriptions...
9,JAM,0.538685,10,personal_remittances_gdp; internet_users_pct; ...,digital_payment_adults_pct; mobile_subscriptio...


### Validación de consistencia

Para cada país-línea, la suma de contribuciones debería ser aproximadamente:
contribution_sum = normalized_values * weights
No necesariamente igual al score TOPSIS.

In [867]:
check = (
    contrib_by_line["PF"]
    .groupby("country_iso3")["contribution"]
    .sum()
    .sort_values(ascending=False)
)

check#.head()

country_iso3
CAN    0.720717
USA    0.713072
ESP    0.703176
CHL    0.643459
URY    0.606650
CRI    0.578843
PAN    0.556241
JAM    0.553253
BRA    0.531795
DOM    0.527727
BHS    0.523927
GUY    0.523496
SUR    0.522206
TTO    0.518906
COL    0.517367
ARG    0.512492
SLV    0.510865
MEX    0.500077
PRY    0.495672
PER    0.491994
BRB    0.482536
BLZ    0.473189
ECU    0.446576
BOL    0.444772
GTM    0.429659
HND    0.405650
NIC    0.368952
VEN    0.367237
HTI    0.214732
Name: contribution, dtype: float64

La contribución aditiva responde:
¿Qué variables explican positivamente el score dado el valor normalizado y el peso?

In [868]:
#Esto genera un “score aditivo aproximado”. compararlo con TOPSIS:
# Si la correlación es alta, la explicación aditiva es razonablemente alineada con el ranking TOPSIS.
pf_ranking = results["business_line_rankings"]["PF"].copy()

if "country_iso3" in pf_ranking.columns:
    pf_ranking = pf_ranking.set_index("country_iso3")

comparison = pd.DataFrame({
    "topsis_score": pf_ranking["score"],
    "additive_contribution_score": check,
})

comparison.corr()

,topsis_score,additive_contribution_score
topsis_score,1.000000,0.941669
additive_contribution_score,0.941669,1.000000


### Analisis marginal

La lógica es:

marginal_effect(variable, país, línea) = score_TOPSIS_completo(país, línea) - score_TOPSIS_sin_variable(país, línea)

Interpretación:

- marginal_effect > 0: la variable ayuda al país en esa línea.
- marginal_effect < 0: la variable lo está penalizando; al quitarla, el score mejora.
- abs(marginal_effect) alto: la variable es materialmente relevante para ese país/línea.
- abs(marginal_effect) cercano a cero: la variable es poco determinante.

No es causalidad. Es una prueba de sensibilidad tipo leave-one-variable-out sobre TOPSIS.

El marginal responde:

Qué tanto cambiaría el score TOPSIS si esta variable no existiera en el modelo?

ejemplo:

- digital_payment_adults_pct puede tener alta contribución en PF, pero si al removerla el ranking casi no cambia, no es un driver robusto.

- country_risk_premium puede tener contribución baja, pero si al removerla mejora mucho el score de un país, es una restricción crítica.


In [869]:

variable_catalog = get_variable_catalog(configs["variables"])

marginal_by_line = compute_all_marginal_effects(
    decision_matrix=decision_matrix,
    weights_audit=weights_audit,
    variable_catalog=variable_catalog,
    distance_metric=configs["settings"]["scoring"]["topsis"].get(
        "distance_metric",
        "euclidean",
    ),
)


2026-06-12 21:41:05 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 29 paises | score max=0.798 (USA)
2026-06-12 21:41:05 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 29 paises | score max=0.798 (USA)
2026-06-12 21:41:05 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 29 paises | score max=0.797 (USA)
2026-06-12 21:41:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 29 paises | score max=0.797 (USA)
2026-06-12 21:41:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 29 paises | score max=0.798 (USA)
2026-06-12 21:41:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 29 paises | score max=0.798 (USA)
2026-06-12 21:41:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 29 paises | score max=0.798 (USA)
2026-06-12 21:41:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 29 paises | score max=0.799 (USA)
2026-06-12 21:41:06 | INFO     | src.scoring.ranking:rank:133 | 

#### Identificar drivers robustos de un país en una línea

Variables con score_effect positivo y alto son drivers robustos.

In [870]:
pf_marginal = marginal_by_line["PF"]

(
    pf_marginal[
        pf_marginal["country_iso3"] == "ARG"
    ]
    .sort_values("score_effect", ascending=False)
    #.head(10)
)

,business_line,country_iso3,removed_variable,dimension,final_topsis_weight,score_full,score_without_variable,score_effect,abs_score_effect,rank_full,rank_without_variable,rank_effect,effect_type,has_override,override_variable_weight_in_dim
876,PF,ARG,digital_payment_adults_pct,digital_tech,0.134400,0.547795,0.502110,0.045685,0.045685,7,16,9,Driver,True,0.32
818,PF,ARG,internet_users_pct,digital_tech,0.067200,0.547795,0.526430,0.021365,0.021365,7,8,1,Driver,True,0.16
847,PF,ARG,mobile_subscriptions,digital_tech,0.092400,0.547795,0.530753,0.017042,0.017042,7,12,5,Driver,True,0.22
354,PF,ARG,account_ownership,financial,0.062526,0.547795,0.534796,0.012999,0.012999,7,8,1,Driver,True,0.22
963,PF,ARG,atms_per_100k_adults,digital_tech,0.046200,0.547795,0.544429,0.003366,0.003366,7,7,0,Driver,True,0.11
383,PF,ARG,interest_rate_spread,financial,0.019895,0.547795,0.546405,0.001389,0.001389,7,7,0,Driver,True,0.07
557,PF,ARG,bank_capital_rwa,financial,0.011368,0.547795,0.547259,0.000536,0.000536,7,7,0,Driver,True,0.04
905,PF,ARG,secure_internet_servers_per_million,digital_tech,0.029400,0.547795,0.547417,0.000378,0.000378,7,8,1,Driver,True,0.07
296,PF,ARG,weighted_mean_applied_tariff_all_products,macro,0.014545,0.547795,0.547467,0.000328,0.000328,7,7,0,Driver,True,0.08
122,PF,ARG,urban_population_pct,macro,0.007273,0.547795,0.547516,0.000279,0.000279,7,7,0,Driver,True,0.04


#### Identificar restricciones robustas


Variables con score_effect negativo son variables que penalizan.
Al removerlas, el score TOPSIS mejora.


In [871]:
(
    pf_marginal[
        pf_marginal["country_iso3"] == "ARG"
    ]
    .sort_values("score_effect", ascending=True)
    #.head(10)
)

,business_line,country_iso3,removed_variable,dimension,final_topsis_weight,score_full,score_without_variable,score_effect,abs_score_effect,rank_full,rank_without_variable,rank_effect,effect_type,has_override,override_variable_weight_in_dim
470,PF,ARG,personal_remittances_gdp,financial,0.079579,0.547795,0.611169,-0.063374,0.063374,7,6,-1,Restriccion,True,0.28
238,PF,ARG,trade_openness,macro,0.040000,0.547795,0.561278,-0.013483,0.013483,7,7,0,Restriccion,True,0.22
528,PF,ARG,financial_system_deposits_gdp,financial,0.028421,0.547795,0.554906,-0.007112,0.007112,7,7,0,Restriccion,True,0.10
64,PF,ARG,inflation_rate,macro,0.029091,0.547795,0.553213,-0.005418,0.005418,7,7,0,Restriccion,True,0.16
325,PF,ARG,domestic_credit_private_gdp,financial,0.028421,0.547795,0.550510,-0.002716,0.002716,7,7,0,Restriccion,True,0.10
992,PF,ARG,ict_goods_exports_pct_total_goods_exports,digital_tech,0.016800,0.547795,0.550228,-0.002433,0.002433,7,7,0,Restriccion,True,0.04
35,PF,ARG,gdp_per_capita_ppp,macro,0.021818,0.547795,0.549338,-0.001543,0.001543,7,7,0,Restriccion,True,0.12
934,PF,ARG,commercial_bank_branches_per_100k_adults,digital_tech,0.033600,0.547795,0.549308,-0.001513,0.001513,7,7,0,Restriccion,True,0.08
412,PF,ARG,bank_npl_ratio,financial,0.022737,0.547795,0.548714,-0.000919,0.000919,7,7,0,Restriccion,True,0.08
731,PF,ARG,control_of_corruption,institutional,0.016900,0.547795,0.548447,-0.000652,0.000652,7,7,0,Restriccion,True,0.13


#### Variables más importantes para una línea completa

Esto muestra qué variables son más materialmente relevantes para la línea, en promedio absoluto.

Interpretación:

- mean_abs_effect: relevancia promedio.
- mean_effect > 0: variable tiende a ayudar en promedio.
- mean_effect < 0: variable tiende a penalizar en promedio.
- max_abs_effect: variable muy importante para algunos países específicos.

In [872]:
line_variable_importance = (
    pf_marginal
    .groupby(["business_line", "dimension", "removed_variable"], as_index=False)
    .agg(
        mean_abs_effect=("abs_score_effect", "mean"),
        mean_effect=("score_effect", "mean"),
        max_abs_effect=("abs_score_effect", "max"),
    )
    .sort_values("mean_abs_effect", ascending=False)
)

line_variable_importance#.head(15)

,business_line,dimension,removed_variable,mean_abs_effect,mean_effect,max_abs_effect
2,PF,digital_tech,digital_payment_adults_pct,0.046269,-0.007342,0.119396
14,PF,financial,personal_remittances_gdp,0.046125,-0.012755,0.110274
5,PF,digital_tech,mobile_subscriptions,0.024660,-0.004929,0.072625
4,PF,digital_tech,internet_users_pct,0.016022,0.013281,0.036565
7,PF,financial,account_ownership,0.008385,0.004429,0.023296
31,PF,macro,trade_openness,0.007269,-0.006191,0.016521
28,PF,macro,inflation_rate,0.006403,0.005708,0.011053
0,PF,digital_tech,atms_per_100k_adults,0.003854,0.002039,0.011625
12,PF,financial,financial_system_deposits_gdp,0.002593,0.000489,0.007337
11,PF,financial,domestic_credit_private_gdp,0.002581,0.001836,0.006238


### Combinar contribución aditiva y efecto marginal

La lectura más robusta aparece cuando cruzas ambas métricas:
- Alta contribución + alto marginal positivo = driver robusto.
- Alta contribución + bajo marginal = driver descriptivo, pero no decisivo.
- Baja contribución + marginal negativo fuerte = restricción crítica.


In [873]:

pf_explainability = combine_contribution_and_marginal(
    contributions=contrib_by_line["PF"],
    marginal_effects=marginal_by_line["PF"],
)


ejemplo para ARG

In [874]:
(
    pf_explainability[
        pf_explainability["country_iso3"] == "ARG"
    ][
        [
            "business_line",
            "country_iso3",
            "removed_variable",
            "dimension_contribution",
            "normalized_value",
            "final_topsis_weight_contribution",
            "contribution",
            "shortfall",
            "score_effect",
            "rank_effect",
            "effect_type",
        ]
    ]
    .sort_values("score_effect", ascending=False)
    #.head(10)
)

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type
0,PF,ARG,digital_payment_adults_pct,digital_tech,0.681100,0.134400,0.091540,0.042860,0.045685,9,Driver
2,PF,ARG,internet_users_pct,digital_tech,0.872843,0.067200,0.058655,0.008545,0.021365,1,Driver
1,PF,ARG,mobile_subscriptions,digital_tech,0.674096,0.092400,0.062286,0.030114,0.017042,5,Driver
3,PF,ARG,account_ownership,financial,0.777665,0.062526,0.048625,0.013902,0.012999,1,Driver
4,PF,ARG,atms_per_100k_adults,digital_tech,0.658154,0.046200,0.030407,0.015793,0.003366,0,Driver
6,PF,ARG,interest_rate_spread,financial,0.807242,0.019895,0.016060,0.003835,0.001389,0,Driver
11,PF,ARG,bank_capital_rwa,financial,0.858999,0.011368,0.009765,0.001603,0.000536,0,Driver
5,PF,ARG,secure_internet_servers_per_million,digital_tech,0.578476,0.029400,0.017007,0.012393,0.000378,1,Driver
13,PF,ARG,weighted_mean_applied_tariff_all_products,macro,0.659466,0.014545,0.009592,0.004953,0.000328,0,Driver
19,PF,ARG,urban_population_pct,macro,0.951911,0.007273,0.006923,0.000350,0.000279,0,Driver


In [875]:
pf_explainability["driver_class"] = pf_explainability.apply(
    classify_driver_robustness,
    axis=1,
)

In [876]:
build_country_driver_table(
    explainability_df=pf_explainability,
    country_iso3="ARG",
    top_n=5,
)

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type,driver_side,driver_class
0,PF,ARG,digital_payment_adults_pct,digital_tech,0.681100,0.134400,0.091540,0.042860,0.045685,9,Driver,Driver,Driver robusto
1,PF,ARG,internet_users_pct,digital_tech,0.872843,0.067200,0.058655,0.008545,0.021365,1,Driver,Driver,Driver robusto
2,PF,ARG,mobile_subscriptions,digital_tech,0.674096,0.092400,0.062286,0.030114,0.017042,5,Driver,Driver,Driver robusto
3,PF,ARG,account_ownership,financial,0.777665,0.062526,0.048625,0.013902,0.012999,1,Driver,Driver,Driver robusto
4,PF,ARG,atms_per_100k_adults,digital_tech,0.658154,0.046200,0.030407,0.015793,0.003366,0,Driver,Driver,Driver descriptivo
5,PF,ARG,personal_remittances_gdp,financial,0.036989,0.079579,0.002944,0.076635,-0.063374,-1,Restriccion,Restriccion,Restriccion critica
6,PF,ARG,trade_openness,macro,0.033014,0.040000,0.001321,0.038679,-0.013483,0,Restriccion,Restriccion,Restriccion critica
7,PF,ARG,financial_system_deposits_gdp,financial,0.000000,0.028421,0.000000,0.028421,-0.007112,0,Restriccion,Restriccion,Driver moderado
8,PF,ARG,inflation_rate,macro,0.137314,0.029091,0.003995,0.025096,-0.005418,0,Restriccion,Restriccion,Driver moderado
9,PF,ARG,domestic_credit_private_gdp,financial,0.323269,0.028421,0.009188,0.019233,-0.002716,0,Restriccion,Restriccion,Efecto marginal bajo


#### Cómo interpretar resultados

Caso 1: contribution alta y score_effect positivo alto
- Interpretación: la variable es un driver robusto. Tiene alto valor normalizado, alto peso y su remoción reduce el score.

Caso 2: contribution alta y score_effect bajo
- Interpretación: la variable describe bien el perfil del país, pero no es decisiva para el TOPSIS.

Caso 3: contribution baja, shortfall alta y score_effect negativo
- Interpretación: la variable es una restricción crítica. El país se beneficiaría si esa dimensión no pesara.

Caso 4: score_effect cercano a cero
- Interpretación: la variable no es material para la posición del país en esa línea.


TOPSIS es no lineal porque depende de:

- distancia al ideal positivo
- distancia al ideal negativo
- normalización relativa
- renormalización de peso

Por tanto: sum(score_effects) != score_full

El marginal es una prueba de robustez, no una atribución contable.

El análisis marginal score_full - score_without_variable complementa la contribución aditiva porque mide robustez, no solo aporte ponderado. Implementa leave-one-variable-out por línea, recalculando TOPSIS sin cada variable y renormalizando pesos. La lectura robusta surge al cruzar contribución, shortfall y efecto marginal: driver robusto, driver descriptivo, restricción crítica o efecto marginal bajo.


